# Chapter 22 — openpyxl: workbook structure, styles, formulas, tables, and charts

**Book role:** Excel workflows  
**Suggested book space:** about 9 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Use openpyxl for workbook features that pandas does not preserve.
- Style headers, formats, and table regions consistently.
- Create formulas and understand cached formula results.
- Generate a polished report while avoiding unsafe workbook assumptions.

## 1. Choose pandas or openpyxl by responsibility

pandas is effective for tabular analysis, type conversion, joins, and group-based summaries. openpyxl manipulates `.xlsx` workbook structure: sheets, cells, styles, formulas, dimensions, merged cells, tables, defined names, and basic charts. They complement one another—compute a summary with pandas, then use openpyxl to add workbook presentation features.

Editing a workbook is not guaranteed to preserve every Excel feature. Advanced shapes, embedded objects, external connections, and some unsupported workbook content can be lost or altered when a library loads and saves the file. Work on a copy of a template, validate the output, and tell readers which workbook features the workflow supports. Avoid using a third-party library to round-trip a business-critical workbook without testing a copy first.

### Worked example 1: Style a worksheet and add an Excel Table

The workbook is reopened to verify its structure. In production, derive the table range from actual data, sanitize table names, and close or release workbook resources after use.

In [ ]:
from pathlib import Path
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.table import Table, TableStyleInfo

output = Path("output") / "styled_sales.xlsx"
output.parent.mkdir(exist_ok=True)
wb = Workbook()
ws = wb.active
ws.title = "Sales"
rows = [
    ["Order ID", "Region", "Revenue", "Profit Margin"],
    [1001, "North", 1250.50, 0.24],
    [1002, "West", 980.00, 0.18],
]
for row in rows:
    ws.append(row)
for cell in ws[1]:
    cell.font = Font(bold=True, color="FFFFFF")
    cell.fill = PatternFill("solid", fgColor="1F4E78")
    cell.alignment = Alignment(horizontal="center")
ws.column_dimensions["A"].width = 14
ws.column_dimensions["B"].width = 16
ws.column_dimensions["C"].width = 16
ws.column_dimensions["D"].width = 18
for cell in ws["C"][1:]:
    cell.number_format = '₹#,##0.00'
for cell in ws["D"][1:]:
    cell.number_format = '0.0%'
table = Table(displayName="SalesTable", ref="A1:D3")
table.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showRowStripes=True, showColumnStripes=False)
ws.add_table(table)
wb.save(output)
check = load_workbook(output, data_only=False)
print(check.sheetnames, check["Sales"].max_row, list(check["Sales"].tables))

## 2. Number formats and table semantics

A stored numeric value and its number format are separate. A cell may contain `1234.5` but display as currency with two decimal places. Set formats for dates, percentages, and monetary columns, and label currency in the workbook. Do not turn numeric cells into preformatted strings if downstream users need to calculate with them.

An Excel Table adds structured references, filter dropdowns, and a defined region. It should have unique column names and a correct range. Tables improve analyst usability for exported rows, but do not guarantee data quality. Validate that row count, column count, and named range match the actual data before saving.

### Worked example 2: Write a formula without expecting Python to calculate it

`data_only=True` reads cached values when present; it does not calculate formulas. A spreadsheet application may be needed to recalculate a workbook. Always validate how the intended recipient will use the report.

In [ ]:
from openpyxl import Workbook, load_workbook
from pathlib import Path

path = Path("output") / "formula_demo.xlsx"
path.parent.mkdir(exist_ok=True)
wb = Workbook()
ws = wb.active
ws.append(["Product", "Revenue"])
ws.append(["A", 100])
ws.append(["B", 250])
ws.append(["Total", "=SUM(B2:B3)"])
wb.save(path)

formula_book = load_workbook(path, data_only=False)
print("Stored formula:", formula_book.active["B4"].value)
value_book = load_workbook(path, data_only=True)
print("Cached result before an external recalculation may be:", value_book.active["B4"].value)

## 3. Formulas and recalculation expectations

openpyxl can write formula strings such as `=SUM(B2:B10)`, but it does not evaluate Excel formulas. Depending on the workbook and consumer, formula results may not be available until Excel or another spreadsheet engine recalculates the workbook. Do not immediately read a value expecting the library to calculate it.

Use formulas when workbook consumers need dynamic spreadsheet behavior; use pandas when a fixed, computed result is sufficient and reproducibility in Python is important. Formula injection is a concern when untrusted text is written into cells: values beginning with characters interpreted as formulas should be treated carefully according to the export requirements and the consuming application.

## Guided lab: Lab: styled analytical workbook

Create a report workbook with a styled title, summary KPI cells, a formatted data table, currency/date/percentage formats, freeze panes, filters, and at least one chart. Use pandas for the calculations and openpyxl for workbook presentation. Reopen the saved file to validate sheet names, dimensions, cell values, formula strings, table range, and chart count.

## Review questions and practice

1. What kinds of work are better suited to openpyxl than pandas?
2. Why should a formula cell not be assumed to have a current calculated value?
3. Why is a number format preferable to converting currency to strings?
4. What risks exist when saving over a sophisticated template workbook?

## Further reading

- openpyxl documentation: https://openpyxl.readthedocs.io/en/stable/
- openpyxl styles: https://openpyxl.readthedocs.io/en/stable/styles.html
- openpyxl charts: https://openpyxl.readthedocs.io/en/stable/charts/introduction.html

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.